# CharmCost — Investigation Notebook
### Metal Cost & Margin Calculator for Charm Jewelry

**Goal:** Investigate feasibility of a tool that compares material cost across 18k gold, 14k gold, and sterling silver for the same charm design, using live spot metal prices, to support a real business decision about which metal(s) to launch a charm line with.

**Core data source:** `xaus.com` — a free, keyless, live-updating spot price API for gold and silver, refreshed continuously and cached ~30 seconds at the edge.

## 1. API research findings

- **Endpoint:** `GET https://xaus.com/api/v1/spot` — no API key, no auth header, no rate limit for reasonable use.
- Returns gold spot price in troy oz *and* already-computed price per gram (`per_gram_usd`) — no unit conversion needed on my end.
- Returns silver spot price per troy oz (`silver_usd_oz`) — I'll need to convert this to per-gram myself (1 troy oz = 31.1035 grams).
- Response also includes a freshness marker (`stale`, `price_as_of`) I can surface to the user so they know how current the price is.
- No documented rate limit, but the docs note "reasonable use" — I'll cache the response for a short window (e.g. 60 seconds) rather than calling on every page load, to be a good API citizen.

**Verified live response (pulled during this research, Aug 9, 2026):**
```json
{
  "xau": {"price": 4343.30, "currency": "USD", "unit": "troy_oz"},
  "per_gram_usd": 139.6403,
  "silver_usd_oz": 63.707001,
  "stale": false,
  "price_as_of": "2026-08-09T07:34:54.655Z",
  "source": "xaus.com"
}
```

This confirms: the API is live, keyless, returns clean JSON, and gives me everything needed for the calculator without a second data source.

In [ ]:
import requests

SPOT_PRICE_URL = "https://xaus.com/api/v1/spot"
TROY_OZ_TO_GRAMS = 31.1034768

def fetch_spot_prices() -> dict:
    """
    Fetch live gold and silver spot prices, in USD per gram of pure (24k / .999 fine) metal.
    """
    response = requests.get(SPOT_PRICE_URL, timeout=15)
    response.raise_for_status()
    data = response.json()

    gold_per_gram = data["per_gram_usd"]
    silver_per_gram = data["silver_usd_oz"] / TROY_OZ_TO_GRAMS

    return {
        "gold_per_gram_usd": round(gold_per_gram, 4),
        "silver_per_gram_usd": round(silver_per_gram, 4),
        "as_of": data.get("price_as_of"),
        "stale": data.get("stale", False),
    }

# Run in Colab (network required):
# prices = fetch_spot_prices()
# prices


## 2. Prototype: karat-based material cost calculator

Standard purity fractions:
- 24k gold = 100% (1.0)
- 18k gold = 75% (0.75)
- 14k gold = 58.3% (0.5833)
- Sterling silver = 92.5% (0.925)

Material cost is simply: `spot_price_per_gram_pure_metal × purity_fraction × charm_weight_grams`

In [ ]:
METAL_OPTIONS = {
    "18k_gold": {"label": "18k Gold", "base": "gold", "purity": 0.75},
    "14k_gold": {"label": "14k Gold", "base": "gold", "purity": 0.5833},
    "sterling_silver": {"label": "Sterling Silver (.925)", "base": "silver", "purity": 0.925},
}

def compare_metals(charm_weight_grams: float, spot_prices: dict) -> list[dict]:
    """
    Return material cost for the same charm design across all metal options,
    sorted from most to least expensive.
    """
    results = []
    for key, meta in METAL_OPTIONS.items():
        base_price = (
            spot_prices["gold_per_gram_usd"] if meta["base"] == "gold"
            else spot_prices["silver_per_gram_usd"]
        )
        price_per_gram = round(base_price * meta["purity"], 4)
        material_cost = round(price_per_gram * charm_weight_grams, 2)
        results.append({
            "metal": meta["label"],
            "price_per_gram": price_per_gram,
            "material_cost": material_cost,
        })
    return sorted(results, key=lambda r: r["material_cost"], reverse=True)

# Example using today's real verified prices from research above:
sample_prices = {"gold_per_gram_usd": 139.6403, "silver_per_gram_usd": 63.707001 / 31.1034768}
for row in compare_metals(charm_weight_grams=3.0, spot_prices=sample_prices):
    print(f"{row['metal']:<24} ${row['price_per_gram']:>8.2f}/g   ->   ${row['material_cost']:>8.2f} for 3.0g charm")


## 3. Prototype: suggested retail price with margin + labor

A charm's total cost isn't just metal -- casting, polishing, and findings (jump rings, clasps) add labor cost. Letting the user enter a flat labor/overhead cost per charm and a target margin gives a full pricing picture, not just a materials comparison.

In [ ]:
def suggested_price(material_cost: float, labor_cost: float, target_margin_pct: float) -> dict:
    """
    Given material + labor cost and a target profit margin (as % of retail price,
    not markup on cost), compute the suggested retail price.

    margin = (price - cost) / price  =>  price = cost / (1 - margin)
    """
    total_cost = material_cost + labor_cost
    margin_fraction = target_margin_pct / 100
    if not 0 <= margin_fraction < 1:
        raise ValueError("target_margin_pct must be between 0 and 100 (exclusive of 100)")

    price = round(total_cost / (1 - margin_fraction), 2)
    profit = round(price - total_cost, 2)

    return {
        "total_cost": round(total_cost, 2),
        "suggested_price": price,
        "profit_per_charm": profit,
    }

# Example: 18k charm, $8 labor, 50% target margin
print(suggested_price(material_cost=314.19, labor_cost=8.00, target_margin_pct=50))


## 4. What I know vs. don't know yet

**Know I know:**
- The spot price API is live, keyless, and returns exactly the units needed (per-gram for gold; per-troy-oz for silver, which is a one-line conversion).
- Karat purity math is simple, well-established, and doesn't require any external lookup -- it's standard jewelry industry constants.
- The margin formula (price = cost / (1 - margin%)) is standard and easy to validate with hand calculations.

**Know I don't know yet:**
- Whether to cache spot prices client-side vs. server-side for the web app, to avoid hitting the API on every keystroke/page load ("reasonable use" isn't precisely defined).
- Whether to support multiple charms per "order" (e.g. a charm bracelet with 5 different charms) in v1, or keep v1 to single-charm comparisons and treat multi-charm as a stretch goal.

**Remaining steps before implementation:**
1. Run the live API call in Colab to confirm real execution (this sandbox has no network access to xaus.com, so code above is written and logically tested but not live-executed).
2. Decide on a simple in-memory cache (e.g. cache spot prices for 60 seconds) before wiring into the Flask app.
3. Confirm rounding/precision choices make sense for a jewelry business context (e.g. round to cents for price, but keep 4 decimal places for price-per-gram).